# Infrastructure Data Cleaning

**Project:** Student Retention & Welfare Tracker  
**Role:** Person 2 — Data Analysis & Data Science

This notebook focuses on inspecting, cleaning, standardizing, and validating
the school infrastructure inspection dataset.

The infrastructure dataset contains periodic inspection records with messy
dates, school IDs, and Boolean values representing the availability or
functionality of school facilities.

The final cleaned dataset will be saved as:

`data/processed/infrastructure_clean.csv`

## Step 1 — Load and Inspect the Infrastructure Data

The raw infrastructure dataset is loaded and inspected before any
transformations are applied.

The initial inspection helps identify the dataset structure, data types,
missing values, duplicate records, and inconsistent representations that
need to be addressed during cleaning.


In [1]:
import pandas as pd
import numpy as np

In [2]:
infrastructure = pd.read_csv(
    "../data/raw/track4_school_infrastructure.csv"
)

In [3]:
print("Shape:", infrastructure.shape)

print("\nColumns:")
print(infrastructure.columns.tolist())

print("\nFirst 5 rows:")
infrastructure.head()

Shape: (3150, 10)

Columns:
['inspection_id', 'date', 'school_id', 'has_electricity', 'has_drinking_water', 'has_functional_toilet', 'has_boundary_wall', 'has_playground', 'inspector_name', 'remarks']

First 5 rows:


,inspection_id,date,school_id,has_electricity,has_drinking_water,has_functional_toilet,has_boundary_wall,has_playground,inspector_name,remarks
0,INSP02966,2025/04/02,sch_0476,H,True,True,Yes,NaN,Fariq Tripathi,Good condition
1,INSP00970,13.05.2025,SCH0337,True,True,Yes,True,H,Unnati Kulkarni,Good condition
2,INSP01386,2025/06/26,SCH0127,True,True,False,Working,Functional,Girik Natt,Toilets locked
3,INSP01234,02-Jun-2025,SCH0372,False,No,True,True,Broken,Rudra Kurian,Good condition
4,INSP02997,2025/10/29,sch0517,True,True,True,True,False,Mohammed Dara,Average


In [4]:
infrastructure.info()

<class 'pandas.DataFrame'>
RangeIndex: 3150 entries, 0 to 3149
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   inspection_id          3150 non-null   str  
 1   date                   3150 non-null   str  
 2   school_id              3150 non-null   str  
 3   has_electricity        2980 non-null   str  
 4   has_drinking_water     3004 non-null   str  
 5   has_functional_toilet  3010 non-null   str  
 6   has_boundary_wall      2973 non-null   str  
 7   has_playground         3004 non-null   str  
 8   inspector_name         2818 non-null   str  
 9   remarks                2637 non-null   str  
dtypes: str(10)
memory usage: 452.3 KB


In [5]:
print("Duplicate rows:", infrastructure.duplicated().sum())

Duplicate rows: 150


## Step 2 — Investigate Duplicate Records

The raw infrastructure dataset contains duplicate rows.

Before removing duplicates, the duplicate records are inspected to determine
whether they represent exact repeated records or legitimate repeated
inspections.

Only confirmed duplicate records will be removed.

In [6]:
duplicate_rows = infrastructure[
    infrastructure.duplicated(keep=False)
].sort_values("inspection_id")

print("Total rows involved in duplicates:", len(duplicate_rows))

duplicate_rows.head(20)

Total rows involved in duplicates: 300


,inspection_id,date,school_id,has_electricity,has_drinking_water,has_functional_toilet,has_boundary_wall,has_playground,inspector_name,remarks
178,INSP00001,2025-09-22,0087,True,True,True,haan,NaN,Ira Ranganathan,Good condition
246,INSP00001,2025-09-22,0087,True,True,True,haan,NaN,Ira Ranganathan,Good condition
192,INSP00015,2025-04-05,SCH0124,True,True,False,H,Y,Yashoda Jani,Average
1502,INSP00015,2025-04-05,SCH0124,True,True,False,H,Y,Yashoda Jani,Average
69,INSP00045,11-04-2025,SCH-0003,Haan,Functional,Broken,1,True,Nicholas Keer,No remarks
1377,INSP00045,11-04-2025,SCH-0003,Haan,Functional,Broken,1,True,Nicholas Keer,No remarks
166,INSP00046,14-Nov-2025,S0280,True,H,True,Y,No,Naksh Ray,Good condition
2551,INSP00046,14-Nov-2025,S0280,True,H,True,Y,No,Naksh Ray,Good condition
111,INSP00052,2025-12-07,SCH-0320,False,True,Yes,Nahi,Kharab,Daksha Kapadia,Good condition
2539,INSP00052,2025-12-07,SCH-0320,False,True,Yes,Nahi,Kharab,Daksha Kapadia,Good condition


In [7]:
print("Duplicate inspection IDs:",
      infrastructure["inspection_id"].duplicated().sum())

Duplicate inspection IDs: 150


In [8]:
print(
    infrastructure[
        infrastructure["inspection_id"].duplicated(keep=False)
    ].sort_values("inspection_id").head(20)
)

     inspection_id         date school_id has_electricity has_drinking_water  \
178      INSP00001   2025-09-22      0087            True               True   
246      INSP00001   2025-09-22      0087            True               True   
192      INSP00015   2025-04-05   SCH0124            True               True   
1502     INSP00015   2025-04-05   SCH0124            True               True   
69       INSP00045   11-04-2025  SCH-0003            Haan         Functional   
1377     INSP00045   11-04-2025  SCH-0003            Haan         Functional   
166      INSP00046  14-Nov-2025     S0280            True                  H   
2551     INSP00046  14-Nov-2025     S0280            True                  H   
111      INSP00052   2025-12-07  SCH-0320           False               True   
2539     INSP00052   2025-12-07  SCH-0320           False               True   
120      INSP00053   2025-04-05   SCH0112            True               True   
2195     INSP00053   2025-04-05   SCH011

### Duplicate Handling Decision

The dataset contains 150 exact duplicate records.

The duplicate records have identical inspection IDs and identical values across
all columns. They therefore represent repeated copies of the same inspection
record rather than separate inspections.

The duplicate rows are removed while retaining the first occurrence of each
record.

In [9]:
# Remove exact duplicate records
infrastructure_clean = infrastructure.drop_duplicates().copy()

print("Rows before removing duplicates:", len(infrastructure))
print("Rows after removing duplicates:", len(infrastructure_clean))
print(
    "Duplicates remaining:",
    infrastructure_clean.duplicated().sum()
)

Rows before removing duplicates: 3150
Rows after removing duplicates: 3000
Duplicates remaining: 0


In [10]:
print(
    "Duplicate inspection IDs remaining:",
    infrastructure_clean["inspection_id"].duplicated().sum()
)

Duplicate inspection IDs remaining: 0


## Step 3 — Missing Value Inspection

After removing exact duplicate records, the cleaned dataset is checked for
missing values.

Missing values are investigated before deciding whether they should be
retained, standardized, or imputed. Infrastructure availability should not
be assumed when an inspection record does not contain a value.

In [11]:
missing_summary = (
    infrastructure_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary

remarks                  488
inspector_name           323
has_boundary_wall        167
has_electricity          164
has_playground           140
has_drinking_water       139
has_functional_toilet    133
inspection_id              0
date                       0
school_id                  0
dtype: int64

In [12]:
print("Total missing values:",
      infrastructure_clean.isna().sum().sum())

Total missing values: 1554


### Missing Value Handling Decision

Missing infrastructure values are retained as missing because an unrecorded
facility status should not be interpreted as either functional or
non-functional.

Similarly, missing inspector names and remarks are retained because these
fields are descriptive and their absence does not prevent infrastructure
analysis.

No rows are removed because of missing values.

## Step 4 — Inspect Infrastructure Status Values

The infrastructure fields contain inconsistent representations of facility
status.

Before standardization, the unique values in each infrastructure field are
examined so that the cleaning rules are based on the actual values present in
the dataset.

In [14]:
for column in infrastructure_columns:
    print(f"\n--- {column} ---")
    print(
        infrastructure_clean[column]
        .value_counts(dropna=False)
        .to_string()
    )


--- has_electricity ---
has_electricity
True             1043
False             371
NaN               164
Y                 149
Yes               137
1                 128
Haan              112
H                 111
haan               99
Hai                94
Working            87
Available          85
Functional         71
N                  54
Nahi hai           53
na                 45
0                  43
No                 41
Nahi               39
Kharab             21
Not Available      19
Under Repair       18
Broken             16

--- has_drinking_water ---
has_drinking_water
True             1211
False             226
1                 175
Y                 168
Yes               150
NaN               139
H                 132
haan              131
Haan              111
Hai               108
Functional         81
Working            79
Available          76
na                 40
0                  34
Nahi               27
No                 26
N                  26
Nahi hai  

In [15]:
for column in infrastructure_columns:
    print(f"\n--- {column} ---")
    values = infrastructure_clean[column].dropna().unique()
    
    for value in sorted(values, key=str):
        print(repr(value))


--- has_electricity ---
'0'
'1'
'Available'
'Broken'
'False'
'Functional'
'H'
'Haan'
'Hai'
'Kharab'
'N'
'Nahi'
'Nahi hai'
'No'
'Not Available'
'True'
'Under Repair'
'Working'
'Y'
'Yes'
'haan'
'na'

--- has_drinking_water ---
'0'
'1'
'Available'
'Broken'
'False'
'Functional'
'H'
'Haan'
'Hai'
'Kharab'
'N'
'Nahi'
'Nahi hai'
'No'
'Not Available'
'True'
'Under Repair'
'Working'
'Y'
'Yes'
'haan'
'na'

--- has_functional_toilet ---
'0'
'1'
'Available'
'Broken'
'False'
'Functional'
'H'
'Haan'
'Hai'
'Kharab'
'N'
'Nahi'
'Nahi hai'
'No'
'Not Available'
'True'
'Under Repair'
'Working'
'Y'
'Yes'
'haan'
'na'

--- has_boundary_wall ---
'0'
'1'
'Available'
'Broken'
'False'
'Functional'
'H'
'Haan'
'Hai'
'Kharab'
'N'
'Nahi'
'Nahi hai'
'No'
'Not Available'
'True'
'Under Repair'
'Working'
'Y'
'Yes'
'haan'
'na'

--- has_playground ---
'0'
'1'
'Available'
'Broken'
'False'
'Functional'
'H'
'Haan'
'Hai'
'Kharab'
'N'
'Nahi'
'Nahi hai'
'No'
'Not Available'
'True'
'Under Repair'
'Working'
'Y'
'Yes'
'haan'
'na'


## Step 5 — Standardize Infrastructure Status Values

The infrastructure fields contain multiple representations of functional and
non-functional status, including English, Hindi-derived terms, abbreviations,
and numeric/Boolean values.

These representations are standardized into:
- `1` = Functional / Available
- `0` = Not functional / Not available
- `NaN` = Missing or unrecorded

Values such as `Under Repair` and `Broken` are treated as non-functional
because the facility cannot currently be considered functional.

Missing values are not converted to `0`, because missing information does not
mean that a facility is unavailable.

In [16]:
infrastructure_status_mapping = {
    "true": 1,
    "false": 0,
    "yes": 1,
    "no": 0,
    "y": 1,
    "n": 0,
    "1": 1,
    "0": 0,
    "haan": 1,
    "hai": 1,
    "h": 1,
    "nahi": 0,
    "nahi hai": 0,
    "working": 1,
    "functional": 1,
    "available": 1,
    "kharab": 0,
    "broken": 0,
    "not available": 0,
    "under repair": 0
}

In [17]:
def standardize_infrastructure_status(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip().lower()
    
    return infrastructure_status_mapping.get(value, np.nan)

In [18]:
for column in infrastructure_columns:
    infrastructure_clean[column] = (
        infrastructure_clean[column]
        .apply(standardize_infrastructure_status)
    )

In [19]:
for column in infrastructure_columns:
    print(f"\n--- {column} ---")
    print(infrastructure_clean[column].value_counts(dropna=False))


--- has_electricity ---
has_electricity
1.0    2116
0.0     675
NaN     209
Name: count, dtype: int64

--- has_drinking_water ---
has_drinking_water
1.0    2422
0.0     399
NaN     179
Name: count, dtype: int64

--- has_functional_toilet ---
has_functional_toilet
1.0    2287
0.0     540
NaN     173
Name: count, dtype: int64

--- has_boundary_wall ---
has_boundary_wall
1.0    1713
0.0    1034
NaN     253
Name: count, dtype: int64

--- has_playground ---
has_playground
1.0    1407
0.0    1341
NaN     252
Name: count, dtype: int64


In [20]:
for column in infrastructure_columns:
    unexpected = infrastructure_clean[
        ~infrastructure_clean[column].isin([0, 1]) &
        infrastructure_clean[column].notna()
    ][column].unique()

    print(f"{column}: {unexpected}")

has_electricity: []
has_drinking_water: []
has_functional_toilet: []
has_boundary_wall: []
has_playground: []


## Step 6 — Date Standardization

The infrastructure inspection dates contain multiple formats, including
different separators and textual month representations.

The dates are converted into a single standardized date format so that
inspection records can be correctly ordered and joined with other datasets.

Invalid or unparseable dates will be identified and validated before the
cleaned dataset is saved.

In [21]:
infrastructure_clean["date_clean"] = pd.to_datetime(
    infrastructure_clean["date"],
    format="mixed",
    dayfirst=False,
    errors="coerce"
)

In [22]:
print(
    "Invalid dates:",
    infrastructure_clean["date_clean"].isna().sum()
)

Invalid dates: 0


In [23]:
infrastructure_clean[["date", "date_clean"]].head(20)

,date,date_clean
0,2025/04/02,2025-04-02
1,13.05.2025,2025-05-13
2,2025/06/26,2025-06-26
3,02-Jun-2025,2025-06-02
4,2025/10/29,2025-10-29
5,10-30-2025,2025-10-30
6,23.07.2025,2025-07-23
7,2025-12-29,2025-12-29
8,2026-01-20,2026-01-20
9,2025/05/12,2025-05-12


## Step 7 — School ID Standardization

School IDs appear in multiple formats, such as `SCH0337`, `sch_0476`,
and other variations.

To ensure that infrastructure records can be reliably joined with the
school master and attendance datasets, all school IDs are converted to
the canonical format `SCH####`.

In [24]:
def standardize_school_id(value):
    value = str(value).strip().upper()
    
    value = value.replace("-", "").replace("_", "")
    
    if value.startswith("SCH"):
        value = value[3:]
    elif value.startswith("S"):
        value = value[1:]
    
    if value.isdigit():
        return "SCH" + value.zfill(4)
    
    return np.nan

In [25]:
infrastructure_clean["school_id_clean"] = (
    infrastructure_clean["school_id"]
    .apply(standardize_school_id)
)

In [26]:
print(
    "Missing standardized school IDs:",
    infrastructure_clean["school_id_clean"].isna().sum()
)

print(
    "Unique standardized school IDs:",
    infrastructure_clean["school_id_clean"].nunique()
)

Missing standardized school IDs: 0
Unique standardized school IDs: 598


In [27]:
infrastructure_clean[
    ["school_id", "school_id_clean"]
].head(20)

,school_id,school_id_clean
0,sch_0476,SCH0476
1,SCH0337,SCH0337
2,SCH0127,SCH0127
3,SCH0372,SCH0372
4,sch0517,SCH0517
5,sch_0334,SCH0334
6,S0229,SCH0229
7,0072,SCH0072
8,SCH0516,SCH0516
9,sch_0182,SCH0182


In [28]:
print(
    "Duplicate inspection IDs:",
    infrastructure_clean["inspection_id"].duplicated().sum()
)

Duplicate inspection IDs: 0


## Step 8 — Prepare the Final Clean Dataset

The temporary columns created during cleaning are now used to replace the
original date and school ID columns.

The final dataset contains standardized dates, school IDs, and infrastructure
status values while retaining descriptive fields such as inspector names and
remarks.

The cleaned dataset is saved to `data/processed/infrastructure_clean.csv`.

In [29]:
infrastructure_final = infrastructure_clean.copy()

infrastructure_final = infrastructure_final.drop(
    columns=["date", "school_id"]
)

infrastructure_final = infrastructure_final.rename(
    columns={
        "date_clean": "date",
        "school_id_clean": "school_id"
    }
)

In [30]:
infrastructure_final.info()

<class 'pandas.DataFrame'>
Index: 3000 entries, 0 to 3149
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   inspection_id          3000 non-null   str           
 1   has_electricity        2791 non-null   float64       
 2   has_drinking_water     2821 non-null   float64       
 3   has_functional_toilet  2827 non-null   float64       
 4   has_boundary_wall      2747 non-null   float64       
 5   has_playground         2748 non-null   float64       
 6   inspector_name         2677 non-null   str           
 7   remarks                2512 non-null   str           
 8   date                   3000 non-null   datetime64[us]
 9   school_id              3000 non-null   str           
dtypes: datetime64[us](1), float64(5), str(4)
memory usage: 366.4 KB


In [31]:
print("Final shape:", infrastructure_final.shape)
print("Duplicate rows:", infrastructure_final.duplicated().sum())
print(
    "Duplicate inspection IDs:",
    infrastructure_final["inspection_id"].duplicated().sum()
)

Final shape: (3000, 10)
Duplicate rows: 0
Duplicate inspection IDs: 0


In [32]:
infrastructure_final.head()

,inspection_id,has_electricity,has_drinking_water,has_functional_toilet,has_boundary_wall,has_playground,inspector_name,remarks,date,school_id
0,INSP02966,1.0,1.0,1.0,1.0,NaN,Fariq Tripathi,Good condition,2025-04-02,SCH0476
1,INSP00970,1.0,1.0,1.0,1.0,1.0,Unnati Kulkarni,Good condition,2025-05-13,SCH0337
2,INSP01386,1.0,1.0,0.0,1.0,1.0,Girik Natt,Toilets locked,2025-06-26,SCH0127
3,INSP01234,0.0,0.0,1.0,1.0,0.0,Rudra Kurian,Good condition,2025-06-02,SCH0372
4,INSP02997,1.0,1.0,1.0,1.0,0.0,Mohammed Dara,Average,2025-10-29,SCH0517


In [33]:
print("Missing values by column:")
print(infrastructure_final.isna().sum())

Missing values by column:
inspection_id              0
has_electricity          209
has_drinking_water       179
has_functional_toilet    173
has_boundary_wall        253
has_playground           252
inspector_name           323
remarks                  488
date                       0
school_id                  0
dtype: int64


## Cleaning Summary and Decisions

| Cleaning Task | Action Taken | Result |
|---|---|---|
| Duplicate records | Removed exact duplicate rows | 150 duplicates removed |
| Missing infrastructure values | Retained as missing | Not treated as non-functional |
| Infrastructure status values | Standardized to 1, 0, or NaN | Consistent Boolean representation |
| Date formats | Converted to standardized dates | 0 invalid dates |
| School IDs | Converted to `SCH####` format | 0 missing standardized IDs |
| Inspection IDs | Checked for duplicates | 0 duplicates remaining |

### Key Cleaning Decisions

- Exact duplicate inspection records were removed because they had identical
  inspection IDs and identical values across all columns.
- Missing infrastructure values were retained rather than converted to `0`,
  because an unrecorded status does not imply that a facility is unavailable.
- Values such as `True`, `Yes`, `Y`, `1`, `Haan`, `Hai`, `Working`,
  `Functional`, and `Available` were standardized to `1`.
- Values such as `False`, `No`, `N`, `0`, `Nahi`, `Nahi hai`, `Kharab`,
  `Broken`, `Not Available`, and `Under Repair` were standardized to `0`.
- The text value `na` was treated as missing rather than as a non-functional
  status.
- Dates were converted into a consistent date representation.
- School IDs were standardized to the canonical `SCH####` format to support
  reliable joins with the other project datasets.

In [34]:
print("INFRASTRUCTURE CLEANING SUMMARY")
print("-" * 40)
print("Raw rows:", len(infrastructure))
print("Cleaned rows:", len(infrastructure_final))
print("Rows removed:", len(infrastructure) - len(infrastructure_final))
print("Columns:", len(infrastructure_final.columns))
print(
    "Duplicate inspection IDs:",
    infrastructure_final["inspection_id"].duplicated().sum()
)
print(
    "Invalid dates:",
    infrastructure_final["date"].isna().sum()
)
print(
    "Missing school IDs:",
    infrastructure_final["school_id"].isna().sum()
)

INFRASTRUCTURE CLEANING SUMMARY
----------------------------------------
Raw rows: 3150
Cleaned rows: 3000
Rows removed: 150
Columns: 10
Duplicate inspection IDs: 0
Invalid dates: 0
Missing school IDs: 0


In [35]:
infrastructure_final.to_csv(
    "../data/processed/infrastructure_clean.csv",
    index=False
)

print("Cleaned infrastructure dataset saved successfully!")

Cleaned infrastructure dataset saved successfully!
